In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, 
    KFold, RepeatedKFold
)
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, 
    mean_absolute_percentage_error
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.inspection import permutation_importance
import lightgbm as lgb
import warnings
import os
import joblib
import json
import sys
import platform
from scipy.stats import pearsonr, spearmanr, sem, t
from scipy import stats
import matplotlib.pyplot as plt
import logging
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
import time

# ============================================================================
# CONFIGURATION
# ============================================================================
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ============================================================================
# REPRODUCIBILITY REPORT
# ============================================================================
def generate_reproducibility_report():
    """Generate comprehensive reproducibility report."""
    report = {
        'timestamp': datetime.now().isoformat(),
        'python_version': sys.version,
        'platform': platform.platform(),
        'packages': {},
        'random_seeds': {
            'main': 42,
            'random_search': 42,
            'cv': 42,
            'train_test_split': 42,
            'repeated_cv': 42,
            'bootstrap': 42,
            'permutation_importance': 42
        }
    }
    
    packages = ['numpy', 'pandas', 'sklearn', 'scipy', 'matplotlib', 'joblib', 'lightgbm']
    
    for pkg in packages:
        try:
            module = __import__(pkg)
            report['packages'][pkg] = module.__version__
        except:
            report['packages'][pkg] = 'not installed'
    
    return report

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def sanitize_feature_names(df: pd.DataFrame) -> pd.DataFrame:
    """Sanitize feature names to remove special characters."""
    import re
    def clean_name(name: str) -> str:
        name = re.sub(r'[()\[\]{}<>/\\|;:,.\'"]+', '_', str(name))
        name = re.sub(r'\s+', '_', name)
        name = re.sub(r'[^a-zA-Z0-9_]', '', name)
        if name and name[0].isdigit():
            name = 'f_' + name
        if not name:
            name = 'feature'
        return name
    
    df.columns = [clean_name(col) for col in df.columns]
    return df

def calculate_nrmse(y_true, y_pred, method='mean'):
    """Calculate normalized RMSE."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    if method == 'mean':
        return rmse / np.mean(y_true)
    elif method == 'range':
        return rmse / (np.max(y_true) - np.min(y_true))
    else:
        return rmse / np.std(y_true)

def bootstrap_metrics(y_true, y_pred, n_bootstrap=1000, random_state=42):
    """Calculate metrics with bootstrap confidence intervals."""
    np.random.seed(random_state)
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    
    metrics = {
        'R2': [],
        'RMSE': [],
        'MAE': [],
        'MAPE': []
    }
    
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        y_true_boot = y_true[idx]
        y_pred_boot = y_pred[idx]
        
        metrics['R2'].append(r2_score(y_true_boot, y_pred_boot))
        metrics['RMSE'].append(np.sqrt(mean_squared_error(y_true_boot, y_pred_boot)))
        metrics['MAE'].append(mean_absolute_error(y_true_boot, y_pred_boot))
        metrics['MAPE'].append(mean_absolute_percentage_error(y_true_boot, y_pred_boot))
    
    results = {}
    for metric_name, values in metrics.items():
        values = np.array(values)
        results[metric_name] = {
            'mean': np.mean(values),
            'std': np.std(values),
            'lower_95': np.percentile(values, 2.5),
            'upper_95': np.percentile(values, 97.5)
        }
    
    return results

def calculate_residual_metrics(y_true, y_pred):
    """Calculate comprehensive residual statistics."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    residuals = y_true - y_pred
    standardized_residuals = residuals / np.std(residuals) if np.std(residuals) > 0 else residuals
    
    return {
        'mean': np.mean(residuals),
        'std': np.std(residuals),
        'min': np.min(residuals),
        'max': np.max(residuals),
        'skew': stats.skew(residuals),
        'kurtosis': stats.kurtosis(residuals),
        'shapiro_wilk_p': stats.shapiro(residuals)[1] if len(residuals) >= 3 else np.nan,
    }

def calculate_ci_from_scores(scores, confidence=0.95):
    """Calculate confidence intervals from a list of scores."""
    scores = np.array(scores)
    n = len(scores)
    mean = np.mean(scores)
    std = np.std(scores, ddof=1)
    se = sem(scores)
    
    t_val = t.ppf((1 + confidence) / 2, n - 1)
    ci_lower = mean - t_val * se
    ci_upper = mean + t_val * se
    
    return {
        'mean': mean,
        'std': std,
        'se': se,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'n': n
    }

# ============================================================================
# PREPROCESSOR - SIMPLIFIED FOR TREE-BASED MODELS
# ============================================================================
class TreePreprocessor(BaseEstimator, TransformerMixin):
    """
    Simple preprocessor for tree-based models.
    - Handles missing values (median imputation)
    - NO feature selection (trees handle this natively)
    - NO scaling (not needed for trees)
    """
    def __init__(self, handle_missing: str = 'median'):
        self.handle_missing = handle_missing
        self.imputer = None
        self.feature_names_in_ = None
        
    def fit(self, X: pd.DataFrame, y: pd.Series = None):
        X_copy = X.copy()
        self.feature_names_in_ = X_copy.columns.tolist()
        
        # Handle missing values
        if self.handle_missing == 'median':
            self.imputer = SimpleImputer(strategy='median')
        elif self.handle_missing == 'mean':
            self.imputer = SimpleImputer(strategy='mean')
        elif self.handle_missing == 'constant':
            self.imputer = SimpleImputer(strategy='constant', fill_value=0)
        else:
            self.imputer = SimpleImputer(strategy='median')
        
        self.imputer.fit(X_copy)
        return self
    
    def transform(self, X: pd.DataFrame):
        X_copy = X.copy()
        
        if self.imputer is not None:
            X_imputed = self.imputer.transform(X_copy)
            X_imputed = pd.DataFrame(X_imputed, columns=X_copy.columns, index=X_copy.index)
        else:
            X_imputed = X_copy
        
        return X_imputed

# ============================================================================
# SCIENTIFIC LIGHTGBM - 80/20 SPLIT WITH REPEATED CV + RANDOM SEARCH
# ============================================================================
class ScientificLightGBM:
    """
    LightGBM with rigorous 80/20 split and repeated cross-validation.
    Uses RandomizedSearchCV for efficient hyperparameter optimization.
    
    WORKFLOW:
    1. Single 80/20 train/test split (stored in model)
    2. RepeatedKFold (5×3 = 15 splits) for hyperparameter tuning
    3. RandomizedSearchCV (100 iterations) for efficient hyperparameter search
    4. Retrain best model on full 80%
    5. Evaluate on held-out 20%
    6. Permutation importance for feature importance
    7. Bootstrap CIs for all metrics
    """
    
    def __init__(
        self,
        config: Dict[str, Any] = None,
        output_dir: str = None
    ):
        self.config = config or self._default_config()
        self.output_dir = output_dir or r"D:\2026 Work\My Papers\Chloride Ingress\Models\LightGBM"
        
        self.pipeline = None
        self.best_params = {}
        self.metrics_train = {}
        self.metrics_test = {}
        self.metrics_test_bootstrap = {}
        self.metrics_cv = {}
        self.metrics_cv_bootstrap = {}
        self.cv_predictions = None
        self.random_search_results = None
        self.original_feature_names = None
        self.reproducibility_info = None
        self.residual_analysis = {}
        self.permutation_importance = None
        
        # Store data splits
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.y_train_pred = None
        self.y_test_pred = None
        
        # Split sizes
        self.n_train_samples = 0
        self.n_test_samples = 0
        self.n_features = 0
        
        self._setup_directories()
        
    def _setup_directories(self):
        for subdir in ['Plots', 'Tables', 'Models', 'Reports', 'Predictions']:
            os.makedirs(os.path.join(self.output_dir, subdir), exist_ok=True)
    
    def _default_config(self) -> Dict[str, Any]:
        return {
            'test_size': 0.20,
            'random_state': 42,
            'cv_folds': 5,
            'cv_repeats': 3,  # 5×3 = 15 splits
            'n_iter': 100,    # Random search iterations
            'n_jobs': -1,
            'cv_scoring': 'neg_root_mean_squared_error',  # Better for interpretation
            'n_bootstrap': 1000,
            'n_permutation_repeats': 30
        }
    
    def _create_pipeline(self, preprocessor=None):
        """Create the LightGBM pipeline."""
        if preprocessor is None:
            preprocessor = TreePreprocessor(handle_missing='median')
        
        # LightGBM model with default params (will be tuned)
        model = lgb.LGBMRegressor(
            n_estimators=100,
            learning_rate=0.1,
            num_leaves=31,
            max_depth=6,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=0.0,
            random_state=self.config['random_state'],
            n_jobs=self.config['n_jobs'],
            verbose=-1
        )
        
        self.pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model)
        ])
        
        return self.pipeline
    
    def _get_random_search_distributions(self) -> Dict:
        """
        RANDOM SEARCH DISTRIBUTIONS FOR LIGHTGBM:
        Uses continuous distributions for more efficient exploration.
        
        With n_iter=100 and 15 folds = 1,500 model fits
        (vs 19,440 for grid search - ~92% faster)
        """
        return {
            'model__n_estimators': [50, 100, 150, 200, 250, 300],
            'model__learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.25, 0.3],
            'model__num_leaves': [10, 15, 20, 25, 31, 40, 50, 63, 80, 100],
            'model__max_depth': [3, 4, 5, 6, 7, 8, 9, 10, 12, 15],
            'model__min_child_samples': [5, 10, 15, 20, 25, 30, 40, 50],
            'model__subsample': [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0],
            'model__colsample_bytree': [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0],
            'model__reg_alpha': [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
            'model__reg_lambda': [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
        }
    
    def _create_repeated_cv(self):
        """Create RepeatedKFold with shuffling for reproducibility."""
        return RepeatedKFold(
            n_splits=self.config['cv_folds'],
            n_repeats=self.config['cv_repeats'],
            random_state=self.config['random_state']
        )
    
    def train(self, X: pd.DataFrame, y: pd.Series, n_iter: int = None):
        """
        Train with 80/20 split and repeated cross-validation using Random Search.
        
        Parameters:
        -----------
        X : pd.DataFrame
            Feature matrix
        y : pd.Series
            Target variable
        n_iter : int
            Number of random search iterations (default: 100)
        """
        logger.info("=" * 80)
        logger.info("SCIENTIFIC LIGHTGBM - 80/20 SPLIT WITH REPEATED CV + RANDOM SEARCH")
        logger.info("=" * 80)
        
        # Set n_iter
        if n_iter is None:
            n_iter = self.config['n_iter']
        
        self.original_feature_names = X.columns.tolist()
        self.n_features = len(X.columns)
        self.reproducibility_info = generate_reproducibility_report()
        
        # ====================================================================
        # STEP 1: SINGLE 80/20 TRAIN/TEST SPLIT (STORED IN MODEL)
        # ====================================================================
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y,
            test_size=self.config['test_size'],
            random_state=self.config['random_state'],
            shuffle=True
        )
        
        self.n_train_samples = len(self.X_train)
        self.n_test_samples = len(self.X_test)
        
        logger.info(f"Data Split (80/20):")
        logger.info(f"  Training: {self.n_train_samples} samples ({self.n_train_samples/(self.n_train_samples+self.n_test_samples):.1%})")
        logger.info(f"  Testing : {self.n_test_samples} samples ({self.n_test_samples/(self.n_train_samples+self.n_test_samples):.1%})")
        
        # ====================================================================
        # STEP 2: REPEATED CROSS-VALIDATION (5×3 = 15 splits) + RANDOM SEARCH
        # ====================================================================
        param_dist = self._get_random_search_distributions()
        total_fits = n_iter * self.config['cv_folds'] * self.config['cv_repeats']
        
        logger.info(f"\nRandomizedSearchCV with RepeatedKFold (5×3 = 15 splits):")
        logger.info(f"  Random iterations: {n_iter}")
        logger.info(f"  Total fits: {n_iter} × {self.config['cv_folds']} × {self.config['cv_repeats']} = {total_fits}")
        logger.info(f"  Estimated time: ~{total_fits / 30:.1f} seconds (with n_jobs=-1)")
        logger.info(f"  ⚡ This is ~{int(19440/total_fits)}x faster than full grid search")
        
        preprocessor = TreePreprocessor(handle_missing='median')
        pipeline = self._create_pipeline(preprocessor)
        
        # Use RepeatedKFold for reproducible CV
        cv = self._create_repeated_cv()
        
        random_search = RandomizedSearchCV(
            pipeline,
            param_distributions=param_dist,
            n_iter=n_iter,
            cv=cv,
            scoring=self.config['cv_scoring'],
            n_jobs=self.config['n_jobs'],
            random_state=self.config['random_state'],
            verbose=0,
            return_train_score=True
        )
        
        start_time = time.time()
        random_search.fit(self.X_train, self.y_train)
        elapsed_time = time.time() - start_time
        
        self.random_search_results = random_search
        self.best_params = random_search.best_params_
        
        logger.info(f"\nRandomizedSearchCV completed in {elapsed_time:.1f} seconds")
        logger.info(f"Best parameters: {self.best_params}")
        logger.info(f"Best CV RMSE: {-random_search.best_score_:.4f}")
        
        # ====================================================================
        # STEP 3: CROSS-VALIDATION METRICS (FOR PUBLICATION)
        # ====================================================================
        logger.info("\nCalculating cross-validation metrics...")
        
        # Get all CV scores from random search results
        cv_results = pd.DataFrame(random_search.cv_results_)
        
        # Get the best model's individual fold scores
        best_index = random_search.best_index_
        
        # Properly extract split scores and convert to numeric
        split_cols = [col for col in cv_results.columns if 'split' in col and 'test_score' in col]
        if split_cols:
            best_cv_scores = cv_results.loc[best_index, split_cols].values
            best_cv_scores = np.array(best_cv_scores, dtype=np.float64)
            best_cv_scores = best_cv_scores[~np.isnan(best_cv_scores)]
            best_cv_rmse = -best_cv_scores  # Since scoring is neg_root_mean_squared_error
        else:
            best_cv_rmse = np.array([-random_search.best_score_])
        
        # ====================================================================
        # Manual cross-validation prediction for RepeatedKFold
        # ====================================================================
        cv = self._create_repeated_cv()
        
        # Manually perform cross-validation predictions
        cv_predictions = np.zeros(len(self.X_train))
        cv_indices = np.zeros(len(self.X_train), dtype=int)
        
        fold_idx = 0
        for train_idx, val_idx in cv.split(self.X_train):
            # Train on train_idx
            X_train_fold = self.X_train.iloc[train_idx]
            y_train_fold = self.y_train.iloc[train_idx]
            
            # Create a fresh pipeline for this fold
            fold_pipeline = self._create_pipeline(preprocessor)
            fold_pipeline.fit(X_train_fold, y_train_fold)
            
            # Predict on val_idx
            X_val_fold = self.X_train.iloc[val_idx]
            y_pred_fold = fold_pipeline.predict(X_val_fold)
            
            # Store predictions
            cv_predictions[val_idx] = y_pred_fold
            cv_indices[val_idx] = fold_idx
            fold_idx += 1
        
        self.cv_predictions = cv_predictions
        
        # Calculate metrics from CV predictions
        self.metrics_cv = {
            'R2_mean': r2_score(self.y_train, cv_predictions),
            'RMSE_mean': np.sqrt(mean_squared_error(self.y_train, cv_predictions)),
            'MAE_mean': mean_absolute_error(self.y_train, cv_predictions)
        }
        
        # Calculate per-fold metrics
        fold_metrics = []
        for fold_i in range(fold_idx):
            mask = cv_indices == fold_i
            if np.sum(mask) > 0:
                y_val_fold = self.y_train[mask]
                y_pred_fold = cv_predictions[mask]
                
                fold_metrics.append({
                    'fold': fold_i,
                    'R2': r2_score(y_val_fold, y_pred_fold),
                    'RMSE': np.sqrt(mean_squared_error(y_val_fold, y_pred_fold)),
                    'MAE': mean_absolute_error(y_val_fold, y_pred_fold)
                })
        
        self.metrics_cv['fold_metrics'] = fold_metrics
        
        # Calculate CV statistics
        cv_r2_scores = [m['R2'] for m in fold_metrics]
        cv_rmse_scores = [m['RMSE'] for m in fold_metrics]
        
        self.metrics_cv['R2_stats'] = calculate_ci_from_scores(cv_r2_scores)
        self.metrics_cv['RMSE_stats'] = calculate_ci_from_scores(cv_rmse_scores)
        
        logger.info(f"\nCross-Validation Metrics (5×3 = 15 folds):")
        logger.info(f"  R²  : {self.metrics_cv['R2_stats']['mean']:.4f} ± {self.metrics_cv['R2_stats']['std']:.4f}")
        logger.info(f"  R² 95% CI: [{self.metrics_cv['R2_stats']['ci_lower']:.4f}, {self.metrics_cv['R2_stats']['ci_upper']:.4f}]")
        logger.info(f"  RMSE: {self.metrics_cv['RMSE_stats']['mean']:.4f} ± {self.metrics_cv['RMSE_stats']['std']:.4f}")
        logger.info(f"  RMSE 95% CI: [{self.metrics_cv['RMSE_stats']['ci_lower']:.4f}, {self.metrics_cv['RMSE_stats']['ci_upper']:.4f}]")
        
        # ====================================================================
        # STEP 4: RETRAIN ON FULL TRAINING SET
        # ====================================================================
        logger.info("\nRetraining best model on full training set...")
        self.pipeline = random_search.best_estimator_
        
        # ====================================================================
        # STEP 5: EVALUATE ON TRAINING SET
        # ====================================================================
        self.y_train_pred = self.pipeline.predict(self.X_train)
        
        self.metrics_train = {
            'R2': r2_score(self.y_train, self.y_train_pred),
            'RMSE': np.sqrt(mean_squared_error(self.y_train, self.y_train_pred)),
            'MAE': mean_absolute_error(self.y_train, self.y_train_pred),
            'MAPE': mean_absolute_percentage_error(self.y_train, self.y_train_pred),
            'NRMSE_mean': calculate_nrmse(self.y_train, self.y_train_pred, method='mean'),
            'Pearson': pearsonr(self.y_train, self.y_train_pred)[0],
            'Spearman': spearmanr(self.y_train, self.y_train_pred)[0]
        }
        
        logger.info(f"\nTraining Set Metrics:")
        for metric, value in self.metrics_train.items():
            logger.info(f"  {metric:15s}: {value:.4f}")
        
        # ====================================================================
        # STEP 6: EVALUATE ON TEST SET
        # ====================================================================
        self.y_test_pred = self.pipeline.predict(self.X_test)
        
        self.metrics_test = {
            'R2': r2_score(self.y_test, self.y_test_pred),
            'RMSE': np.sqrt(mean_squared_error(self.y_test, self.y_test_pred)),
            'MAE': mean_absolute_error(self.y_test, self.y_test_pred),
            'MAPE': mean_absolute_percentage_error(self.y_test, self.y_test_pred),
            'NRMSE_mean': calculate_nrmse(self.y_test, self.y_test_pred, method='mean'),
            'NRMSE_range': calculate_nrmse(self.y_test, self.y_test_pred, method='range'),
            'Pearson': pearsonr(self.y_test, self.y_test_pred)[0],
            'Spearman': spearmanr(self.y_test, self.y_test_pred)[0]
        }
        
        logger.info(f"\nTest Set Metrics:")
        for metric, value in self.metrics_test.items():
            logger.info(f"  {metric:15s}: {value:.4f}")
        
        # ====================================================================
        # STEP 7: BOOTSTRAP CONFIDENCE INTERVALS
        # ====================================================================
        logger.info("\nCalculating bootstrap confidence intervals...")
        
        # Test set bootstrap
        self.metrics_test_bootstrap = bootstrap_metrics(
            self.y_test, self.y_test_pred,
            n_bootstrap=self.config['n_bootstrap'],
            random_state=self.config['random_state']
        )
        
        logger.info(f"\nTest Set Bootstrap 95% Confidence Intervals:")
        for metric, values in self.metrics_test_bootstrap.items():
            logger.info(f"  {metric:15s}: {values['mean']:.4f} [{values['lower_95']:.4f}, {values['upper_95']:.4f}]")
        
        # CV bootstrap (using CV predictions)
        self.metrics_cv_bootstrap = bootstrap_metrics(
            self.y_train, self.cv_predictions,
            n_bootstrap=self.config['n_bootstrap'],
            random_state=self.config['random_state'] + 1
        )
        
        logger.info(f"\nCV Bootstrap 95% Confidence Intervals:")
        for metric, values in self.metrics_cv_bootstrap.items():
            logger.info(f"  {metric:15s}: {values['mean']:.4f} [{values['lower_95']:.4f}, {values['upper_95']:.4f}]")
        
        # ====================================================================
        # STEP 8: RESIDUAL ANALYSIS
        # ====================================================================
        self.residual_analysis = {
            'test': calculate_residual_metrics(self.y_test, self.y_test_pred),
            'training': calculate_residual_metrics(self.y_train, self.y_train_pred),
            'cv': calculate_residual_metrics(self.y_train, self.cv_predictions)
        }
        
        # ====================================================================
        # STEP 9: PERMUTATION IMPORTANCE (FOR PUBLICATION)
        # ====================================================================
        logger.info("\nCalculating permutation importance...")
        
        try:
            perm_result = permutation_importance(
                self.pipeline,
                self.X_test,
                self.y_test,
                n_repeats=self.config['n_permutation_repeats'],
                random_state=self.config['random_state'],
                scoring='r2',
                n_jobs=self.config['n_jobs']
            )
            
            # Get feature names
            try:
                feature_names = self.pipeline.named_steps['preprocessor'].feature_names_in_
            except:
                feature_names = self.original_feature_names
            
            self.permutation_importance = pd.DataFrame({
                'Feature': feature_names[:len(perm_result.importances_mean)],
                'Importance_mean': perm_result.importances_mean,
                'Importance_std': perm_result.importances_std
            }).sort_values('Importance_mean', ascending=False)
            
            logger.info(f"  Top 5 features by permutation importance:")
            for i, row in self.permutation_importance.head(5).iterrows():
                logger.info(f"    {row['Feature']:30s}: {row['Importance_mean']:.4f} ± {row['Importance_std']:.4f}")
                
        except Exception as e:
            logger.warning(f"Permutation importance failed: {e}")
            self.permutation_importance = None
        
        # ====================================================================
        # STEP 10: SAVE PREDICTIONS AND DATA
        # ====================================================================
        self._save_predictions()
        
        # ====================================================================
        # STEP 11: SUMMARY
        # ====================================================================
        logger.info("\n" + "=" * 80)
        logger.info("TRAINING COMPLETE - SUMMARY")
        logger.info("=" * 80)
        logger.info(f"  Training samples: {self.n_train_samples}")
        logger.info(f"  Test samples: {self.n_test_samples}")
        logger.info(f"  Features: {self.n_features}")
        logger.info(f"  CV R²: {self.metrics_cv['R2_stats']['mean']:.4f} ± {self.metrics_cv['R2_stats']['std']:.4f}")
        logger.info(f"  Test R²: {self.metrics_test['R2']:.4f}")
        logger.info(f"  Test RMSE: {self.metrics_test['RMSE']:.4f}")
        logger.info(f"  Best CV RMSE: {-random_search.best_score_:.4f}")
        logger.info(f"  Random iterations: {n_iter}")
        logger.info(f"  CV splits: {self.config['cv_folds']}×{self.config['cv_repeats']} = {self.config['cv_folds'] * self.config['cv_repeats']}")
        logger.info(f"  Total CV fits: {total_fits}")
        logger.info("=" * 80)
        
        return self
    
    def _save_predictions(self):
        """Save predictions and actual values to CSV."""
        # Training predictions
        train_df = pd.DataFrame({
            'Actual': self.y_train.values if hasattr(self.y_train, 'values') else self.y_train,
            'Predicted': self.y_train_pred,
            'Residual': self.y_train.values if hasattr(self.y_train, 'values') else self.y_train - self.y_train_pred,
            'Set': 'Training'
        })
        
        # Test predictions
        test_df = pd.DataFrame({
            'Actual': self.y_test.values if hasattr(self.y_test, 'values') else self.y_test,
            'Predicted': self.y_test_pred,
            'Residual': self.y_test.values if hasattr(self.y_test, 'values') else self.y_test - self.y_test_pred,
            'Set': 'Test'
        })
        
        # Combine
        all_predictions = pd.concat([train_df, test_df], ignore_index=True)
        
        # Save
        pred_path = os.path.join(self.output_dir, 'Predictions', 'predictions.csv')
        all_predictions.to_csv(pred_path, index=False)
        logger.info(f"Predictions saved to: {pred_path}")
        
        # Save separate files
        train_path = os.path.join(self.output_dir, 'Predictions', 'training_predictions.csv')
        test_path = os.path.join(self.output_dir, 'Predictions', 'test_predictions.csv')
        train_df.to_csv(train_path, index=False)
        test_df.to_csv(test_path, index=False)
        
        # Save with features
        X_train_with_pred = self.X_train.copy()
        X_train_with_pred['Actual'] = self.y_train.values if hasattr(self.y_train, 'values') else self.y_train
        X_train_with_pred['Predicted'] = self.y_train_pred
        X_train_with_pred['Residual'] = X_train_with_pred['Actual'] - X_train_with_pred['Predicted']
        
        X_test_with_pred = self.X_test.copy()
        X_test_with_pred['Actual'] = self.y_test.values if hasattr(self.y_test, 'values') else self.y_test
        X_test_with_pred['Predicted'] = self.y_test_pred
        X_test_with_pred['Residual'] = X_test_with_pred['Actual'] - X_test_with_pred['Predicted']
        
        X_train_with_pred.to_csv(os.path.join(self.output_dir, 'Predictions', 'training_data_with_predictions.csv'), index=False)
        X_test_with_pred.to_csv(os.path.join(self.output_dir, 'Predictions', 'test_data_with_predictions.csv'), index=False)
    
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Make predictions."""
        return self.pipeline.predict(X)
    
    def get_feature_importance(self, method='permutation') -> pd.DataFrame:
        """
        Get feature importance.
        
        Parameters:
        -----------
        method : str
            'permutation' - Permutation importance (preferred for publication)
            'gain' - LightGBM gain-based importance
            'split' - LightGBM split-based importance
        """
        if method == 'permutation' and self.permutation_importance is not None:
            return self.permutation_importance
        elif method in ['gain', 'split']:
            try:
                model = self.pipeline.named_steps['model']
                
                # Get LightGBM feature importance
                importance = model.feature_importances_
                
                try:
                    feature_names = self.pipeline.named_steps['preprocessor'].feature_names_in_
                except:
                    feature_names = self.original_feature_names
                
                if len(importance) != len(feature_names):
                    feature_names = [f'Feature_{i}' for i in range(len(importance))]
                
                return pd.DataFrame({
                    'Feature': feature_names[:len(importance)],
                    'Importance': importance
                }).sort_values('Importance', ascending=False)
            except Exception as e:
                logger.warning(f"Could not extract LightGBM importance: {e}")
                return pd.DataFrame()
        else:
            return pd.DataFrame()
    
    def save_model(self, filepath: str = None):
        """Save the model."""
        if filepath is None:
            filepath = os.path.join(self.output_dir, 'Models', 'LightGBM_80_20.joblib')
        
        model_to_save = {
            'pipeline': self.pipeline,
            'best_params': self.best_params,
            'metrics_train': self.metrics_train,
            'metrics_test': self.metrics_test,
            'metrics_test_bootstrap': self.metrics_test_bootstrap,
            'metrics_cv': self.metrics_cv,
            'metrics_cv_bootstrap': self.metrics_cv_bootstrap,
            'permutation_importance': self.permutation_importance,
            'config': self.config,
            'reproducibility_info': self.reproducibility_info,
            'n_train_samples': self.n_train_samples,
            'n_test_samples': self.n_test_samples,
            'n_features': self.n_features,
            'original_feature_names': self.original_feature_names,
            'random_search_results': self.random_search_results,
            'y_train_pred': self.y_train_pred,
            'y_test_pred': self.y_test_pred
        }
        
        joblib.dump(model_to_save, filepath)
        logger.info(f"Model saved to: {filepath}")
        
        return filepath

# ============================================================================
# VISUALIZER
# ============================================================================
class ScientificVisualizer:
    def __init__(self, output_dir: str, dpi: int = 300):
        self.output_dir = output_dir
        self.plots_dir = os.path.join(output_dir, "Plots")
        self.dpi = dpi
        os.makedirs(self.plots_dir, exist_ok=True)
        
        plt.style.use('seaborn-v0_8-whitegrid')
        self.colors = {
            'blue': '#2E86AB',
            'red': '#D62828',
            'green': '#2E9F6E',
            'purple': '#7B2D8B',
            'orange': '#F77F00',
            'grey': '#6C6C6C'
        }
    
    def generate_all_plots(self, model: ScientificLightGBM):
        """Generate all publication-quality plots."""
        logger.info("Generating publication-quality plots...")
        
        y_train_pred = model.y_train_pred
        y_test_pred = model.y_test_pred
        
        # Figure 1: Scatter plot - Training Set
        self._plot_scatter(model.y_train, y_train_pred, 
                          'Training Set: Actual vs Predicted',
                          os.path.join(self.plots_dir, 'Fig1_Scatter_Training.png'),
                          model.metrics_train)
        
        # Figure 2: Scatter plot - Test Set
        self._plot_scatter(model.y_test, y_test_pred, 
                          'Test Set: Actual vs Predicted',
                          os.path.join(self.plots_dir, 'Fig2_Scatter_Test.png'),
                          model.metrics_test)
        
        # Figure 3: Combined Scatter plot (Training + Test)
        self._plot_combined_scatter(model.y_train, y_train_pred, 
                                   model.y_test, y_test_pred,
                                   os.path.join(self.plots_dir, 'Fig3_Scatter_Combined.png'))
        
        # Figure 4: Scatter plot - CV Predictions
        if hasattr(model, 'cv_predictions') and model.cv_predictions is not None:
            self._plot_scatter(model.y_train, model.cv_predictions, 
                              'Cross-Validation: Actual vs Predicted',
                              os.path.join(self.plots_dir, 'Fig4_Scatter_CV.png'),
                              {'R2': model.metrics_cv['R2_mean'], 
                               'RMSE': model.metrics_cv['RMSE_mean'],
                               'MAE': model.metrics_cv['MAE_mean']})
        
        # Figure 5: Residual analysis - Training Set
        self._plot_residuals(model.y_train, y_train_pred,
                            'Training Set Residual Analysis',
                            os.path.join(self.plots_dir, 'Fig5_Residuals_Training.png'))
        
        # Figure 6: Residual analysis - Test Set
        self._plot_residuals(model.y_test, y_test_pred,
                            'Test Set Residual Analysis',
                            os.path.join(self.plots_dir, 'Fig6_Residuals_Test.png'))
        
        # Figure 7: Residual analysis - CV
        if hasattr(model, 'cv_predictions') and model.cv_predictions is not None:
            self._plot_residuals(model.y_train, model.cv_predictions,
                                'Cross-Validation Residual Analysis',
                                os.path.join(self.plots_dir, 'Fig7_Residuals_CV.png'))
        
        # Figure 8: Q-Q plot - Training Set
        self._plot_qq(model.y_train, y_train_pred,
                     os.path.join(self.plots_dir, 'Fig8_QQ_Training.png'))
        
        # Figure 9: Q-Q plot - Test Set
        self._plot_qq(model.y_test, y_test_pred,
                     os.path.join(self.plots_dir, 'Fig9_QQ_Test.png'))
        
        # Figure 10: Q-Q plot - CV
        if hasattr(model, 'cv_predictions') and model.cv_predictions is not None:
            self._plot_qq(model.y_train, model.cv_predictions,
                         os.path.join(self.plots_dir, 'Fig10_QQ_CV.png'))
        
        # Figure 11: Feature importance (Permutation)
        importance_df = model.get_feature_importance(method='permutation')
        if importance_df is not None and not importance_df.empty:
            self._plot_feature_importance(importance_df.head(15),
                                         os.path.join(self.plots_dir, 'Fig11_Permutation_Importance.png'))
        
        # Figure 12: Random Search Convergence
        if model.random_search_results is not None:
            self._plot_random_search_convergence(model,
                                               os.path.join(self.plots_dir, 'Fig12_RandomSearch_Convergence.png'))
        
        # Figure 13: CV Fold Performance
        if 'fold_metrics' in model.metrics_cv:
            self._plot_cv_folds(model.metrics_cv['fold_metrics'],
                               os.path.join(self.plots_dir, 'Fig13_CV_Folds.png'))
        
        # Figure 14: LightGBM Feature Importance (Gain)
        gain_importance = model.get_feature_importance(method='gain')
        if not gain_importance.empty:
            self._plot_lightgbm_importance(gain_importance.head(15),
                                          os.path.join(self.plots_dir, 'Fig14_LightGBM_Gain_Importance.png'),
                                          importance_type='Gain')
        
        # Figure 15: LightGBM Feature Importance (Split)
        split_importance = model.get_feature_importance(method='split')
        if not split_importance.empty:
            self._plot_lightgbm_importance(split_importance.head(15),
                                          os.path.join(self.plots_dir, 'Fig15_LightGBM_Split_Importance.png'),
                                          importance_type='Split')
        
        # Figure 16: Error Distribution Comparison (Training vs Test)
        self._plot_error_distribution(model.y_train, y_train_pred,
                                     model.y_test, y_test_pred,
                                     os.path.join(self.plots_dir, 'Fig16_Error_Distribution.png'))
        
        # Figure 17: Metrics Comparison Bar Chart
        self._plot_metrics_comparison(model.metrics_train, model.metrics_test,
                                     os.path.join(self.plots_dir, 'Fig17_Metrics_Comparison.png'))
        
        logger.info(f"Plots saved to: {self.plots_dir}")
    
    def _plot_scatter(self, y_true, y_pred, title, filename, metrics=None):
        fig, ax = plt.subplots(figsize=(8, 8))
        
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        
        ax.scatter(y_true, y_pred, c=self.colors['blue'], s=40, alpha=0.6,
                  edgecolors='black', linewidth=0.5)
        
        min_val = min(y_true.min(), y_pred.min()) - 0.1
        max_val = max(y_true.max(), y_pred.max()) + 0.1
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5)
        
        if metrics is None:
            r2 = r2_score(y_true, y_pred)
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mae = mean_absolute_error(y_true, y_pred)
        else:
            r2 = metrics.get('R2', r2_score(y_true, y_pred))
            rmse = metrics.get('RMSE', np.sqrt(mean_squared_error(y_true, y_pred)))
            mae = metrics.get('MAE', mean_absolute_error(y_true, y_pred))
        
        textstr = f'$R^2$ = {r2:.4f}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}'
        props = dict(boxstyle='round', facecolor='white', edgecolor='black', linewidth=1.5, alpha=0.9)
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
                verticalalignment='top', bbox=props)
        
        ax.set_xlabel('Actual Total Chloride (% of binder mass)', fontsize=12)
        ax.set_ylabel('Predicted Total Chloride (% of binder mass)', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_combined_scatter(self, y_train, y_train_pred, y_test, y_test_pred, filename):
        fig, ax = plt.subplots(figsize=(8, 8))
        
        y_train = np.array(y_train)
        y_train_pred = np.array(y_train_pred)
        y_test = np.array(y_test)
        y_test_pred = np.array(y_test_pred)
        
        # Plot training points
        ax.scatter(y_train, y_train_pred, c=self.colors['blue'], s=40, alpha=0.6,
                  edgecolors='black', linewidth=0.5, label='Training')
        
        # Plot test points
        ax.scatter(y_test, y_test_pred, c=self.colors['red'], s=40, alpha=0.6,
                  edgecolors='black', linewidth=0.5, label='Test')
        
        min_val = min(np.concatenate([y_train, y_test, y_train_pred, y_test_pred])) - 0.1
        max_val = max(np.concatenate([y_train, y_test, y_train_pred, y_test_pred])) + 0.1
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5)
        
        # Calculate metrics
        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        
        textstr = f'Training $R^2$ = {train_r2:.4f}\nTest $R^2$ = {test_r2:.4f}'
        props = dict(boxstyle='round', facecolor='white', edgecolor='black', linewidth=1.5, alpha=0.9)
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
                verticalalignment='top', bbox=props)
        
        ax.set_xlabel('Actual Total Chloride (% of binder mass)', fontsize=12)
        ax.set_ylabel('Predicted Total Chloride (% of binder mass)', fontsize=12)
        ax.set_title('Combined: Training vs Test Set Performance', fontsize=14, fontweight='bold')
        ax.legend(loc='lower right')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_residuals(self, y_true, y_pred, title, filename):
        fig, ax = plt.subplots(figsize=(10, 6))
        
        residuals = y_true - y_pred
        fitted = y_pred
        
        ax.scatter(fitted, residuals, c=self.colors['blue'], s=40, alpha=0.6,
                  edgecolors='black', linewidth=0.5)
        
        ax.axhline(y=0, color=self.colors['red'], linestyle='--', linewidth=1.5)
        
        mean_res = np.mean(residuals)
        std_res = np.std(residuals)
        
        textstr = f'Mean: {mean_res:.4f}\nStd: {std_res:.4f}'
        props = dict(boxstyle='round', facecolor='white', edgecolor='black', linewidth=1.5, alpha=0.9)
        ax.text(0.95, 0.95, textstr, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', ha='right', bbox=props)
        
        ax.set_xlabel('Predicted Total Chloride (% of binder mass)', fontsize=12)
        ax.set_ylabel('Residual (Actual - Predicted)', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_qq(self, y_true, y_pred, filename):
        residuals = y_true - y_pred
        standardized_residuals = (residuals - np.mean(residuals)) / np.std(residuals)
        
        fig, ax = plt.subplots(figsize=(8, 8))
        
        stats.probplot(standardized_residuals, dist="norm", plot=ax)
        
        ax.set_title('Q-Q Plot of Standardized Residuals', fontsize=14, fontweight='bold')
        ax.set_xlabel('Theoretical Quantiles', fontsize=12)
        ax.set_ylabel('Sample Quantiles', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        if len(residuals) >= 3:
            _, p_value = stats.shapiro(residuals)
            ax.text(0.05, 0.95, f'Shapiro-Wilk p = {p_value:.4f}', 
                   transform=ax.transAxes, fontsize=10,
                   verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_feature_importance(self, importance_df, filename):
        fig, ax = plt.subplots(figsize=(10, 8))
        
        importance_df = importance_df.head(15)
        
        colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(importance_df)))[::-1]
        
        if 'Importance_std' in importance_df.columns:
            ax.barh(importance_df['Feature'], importance_df['Importance_mean'], 
                   xerr=importance_df['Importance_std'],
                   color=colors, edgecolor='black', linewidth=0.5,
                   capsize=3)
            xlabel = 'Permutation Importance (R² decrease)'
        else:
            ax.barh(importance_df['Feature'], importance_df['Importance'], 
                   color=colors, edgecolor='black', linewidth=0.5)
            xlabel = 'Feature Importance'
        
        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel('Feature', fontsize=12)
        ax.set_title('Top 15 Feature Importances (Permutation)', fontsize=14, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_lightgbm_importance(self, importance_df, filename, importance_type='Gain'):
        fig, ax = plt.subplots(figsize=(10, 8))
        
        importance_df = importance_df.head(15)
        
        colors = plt.cm.Greens(np.linspace(0.4, 0.9, len(importance_df)))[::-1]
        
        ax.barh(importance_df['Feature'], importance_df['Importance'], 
               color=colors, edgecolor='black', linewidth=0.5)
        
        ax.set_xlabel(f'LightGBM {importance_type} Importance', fontsize=12)
        ax.set_ylabel('Feature', fontsize=12)
        ax.set_title(f'Top 15 Features by LightGBM {importance_type} Importance', 
                    fontsize=14, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_random_search_convergence(self, model, filename):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        results = model.random_search_results
        cv_results = pd.DataFrame(results.cv_results_)
        
        # Plot 1: RMSE over iterations
        iterations = np.arange(1, len(cv_results) + 1)
        mean_scores = -cv_results['mean_test_score']
        
        ax1.plot(iterations, mean_scores, 'o-', color=self.colors['blue'], 
                linewidth=2, markersize=4, alpha=0.7)
        ax1.axhline(y=-results.best_score_, color=self.colors['red'], 
                   linestyle='--', linewidth=2, label=f'Best: {-results.best_score_:.4f}')
        
        window = min(10, len(mean_scores))
        if window > 1:
            rolling_avg = pd.Series(mean_scores).rolling(window).mean()
            ax1.plot(iterations, rolling_avg, color=self.colors['green'], 
                    linewidth=2, alpha=0.7, label=f'{window}-point rolling avg')
        
        ax1.set_xlabel('Random Search Iteration', fontsize=12)
        ax1.set_ylabel('RMSE (CV)', fontsize=12)
        ax1.set_title('Random Search Convergence', fontsize=12, fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)
        
        # Plot 2: Distribution of best parameters
        top_params = cv_results.nlargest(10, 'mean_test_score')
        top_indices = top_params.index
        
        n_estimators_top = [cv_results.loc[i, 'params']['model__n_estimators'] for i in top_indices]
        
        ax2.bar(range(len(n_estimators_top)), n_estimators_top, 
               color=plt.cm.Blues(np.linspace(0.4, 0.9, len(n_estimators_top))),
               edgecolor='black', linewidth=0.5)
        ax2.set_xlabel('Top 10 Iterations (by RMSE)', fontsize=12)
        ax2.set_ylabel('n_estimators', fontsize=12)
        ax2.set_title('Top 10: n_estimators Distribution', fontsize=12, fontweight='bold')
        ax2.set_xticks(range(len(n_estimators_top)))
        ax2.set_xticklabels([f'{i+1}' for i in range(len(n_estimators_top))])
        ax2.grid(True, alpha=0.3, axis='y')
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(False)
        
        plt.suptitle('RandomizedSearchCV Convergence Analysis', 
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_cv_folds(self, fold_metrics, filename):
        fig, ax = plt.subplots(figsize=(10, 6))
        
        fold_numbers = [m['fold'] + 1 for m in fold_metrics]
        r2_scores = [m['R2'] for m in fold_metrics]
        rmse_scores = [m['RMSE'] for m in fold_metrics]
        
        x = np.arange(len(fold_numbers))
        width = 0.35
        
        ax.bar(x - width/2, r2_scores, width, label='R²', color=self.colors['blue'], alpha=0.7)
        
        ax2 = ax.twinx()
        ax2.bar(x + width/2, rmse_scores, width, label='RMSE', color=self.colors['red'], alpha=0.7)
        
        ax.axhline(y=np.mean(r2_scores), color='blue', linestyle='--', linewidth=1.5, alpha=0.7)
        ax2.axhline(y=np.mean(rmse_scores), color='red', linestyle='--', linewidth=1.5, alpha=0.7)
        
        ax.set_xlabel('CV Fold', fontsize=12)
        ax.set_ylabel('R²', fontsize=12, color='blue')
        ax2.set_ylabel('RMSE', fontsize=12, color='red')
        ax.set_title('Cross-Validation Performance by Fold (5×3 = 15 folds)', 
                    fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(fold_numbers)
        
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(True)
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_error_distribution(self, y_train, y_train_pred, y_test, y_test_pred, filename):
        fig, ax = plt.subplots(figsize=(10, 6))
        
        train_residuals = y_train - y_train_pred
        test_residuals = y_test - y_test_pred
        
        # Histogram of residuals
        ax.hist(train_residuals, bins=20, alpha=0.5, label='Training', 
                color=self.colors['blue'], edgecolor='black', linewidth=0.5)
        ax.hist(test_residuals, bins=20, alpha=0.5, label='Test', 
                color=self.colors['red'], edgecolor='black', linewidth=0.5)
        
        ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
        
        ax.set_xlabel('Residual (Actual - Predicted)', fontsize=12)
        ax.set_ylabel('Frequency', fontsize=12)
        ax.set_title('Error Distribution: Training vs Test', fontsize=14, fontweight='bold')
        ax.legend()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_metrics_comparison(self, metrics_train, metrics_test, filename):
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Select metrics to compare
        metrics_to_plot = ['R2', 'RMSE', 'MAE', 'MAPE']
        x = np.arange(len(metrics_to_plot))
        width = 0.35
        
        train_values = [metrics_train.get(m, 0) for m in metrics_to_plot]
        test_values = [metrics_test.get(m, 0) for m in metrics_to_plot]
        
        bars1 = ax.bar(x - width/2, train_values, width, label='Training', 
                       color=self.colors['blue'], alpha=0.7, edgecolor='black', linewidth=0.5)
        bars2 = ax.bar(x + width/2, test_values, width, label='Test', 
                       color=self.colors['red'], alpha=0.7, edgecolor='black', linewidth=0.5)
        
        ax.set_xlabel('Metrics', fontsize=12)
        ax.set_ylabel('Value', fontsize=12)
        ax.set_title('Performance Metrics: Training vs Test', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(metrics_to_plot)
        ax.legend()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for bar in bars1:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
        for bar in bars2:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()

# ============================================================================
# MAIN TRAINER
# ============================================================================
class ScientificModelTrainer:
    def __init__(self, config: Dict[str, Any] = None):
        self.config = config or {}
        self.results_dir = r"D:\2026 Work\My Papers\Chloride Ingress\Models\LightGBM"
        
        for subdir in ['Plots', 'Tables', 'Models', 'Reports', 'Predictions']:
            os.makedirs(os.path.join(self.results_dir, subdir), exist_ok=True)
        
        self.model = None
        self.visualizer = ScientificVisualizer(self.results_dir)
    
    def load_data(self, filepath: str) -> Tuple[pd.DataFrame, pd.Series, str]:
        logger.info(f"Loading data from: {filepath}")
        
        df = pd.read_csv(filepath, encoding='ISO-8859-1')
        df.columns = df.columns.str.strip()
        
        logger.info(f"Initial data shape: {df.shape}")
        
        target_col = 'Total chloride, % of binder mass'
        
        if target_col not in df.columns:
            available_cols = df.columns.tolist()
            raise ValueError(
                f"Target column '{target_col}' not found.\n"
                f"Available columns: {available_cols}\n"
                f"Please check the column name."
            )
        
        # Remove exposure zone columns (data leakage)
        features_to_drop = ['Exposure_zones', 'Exposure Zones', 'ExposureZone', 'exposure_zones']
        for feature in features_to_drop:
            if feature in df.columns:
                df = df.drop(columns=[feature])
                logger.warning(f"REMOVED '{feature}' to prevent data leakage")
        
        X = df.drop(columns=[target_col])
        y = df[target_col]
        
        # Handle NaN in target
        nan_count_y = y.isna().sum()
        if nan_count_y > 0:
            logger.warning(f"Found {nan_count_y} NaN values in target")
            valid_idx = ~y.isna()
            X = X[valid_idx]
            y = y[valid_idx]
            logger.info(f"Removed {nan_count_y} rows. New shape: {X.shape}")
        
        X = sanitize_feature_names(X)
        
        logger.info(f"Loaded {len(X)} samples with {len(X.columns)} features")
        logger.info(f"Target stats - Min: {y.min():.4f}, Max: {y.max():.4f}, Mean: {y.mean():.4f}")
        
        return X, y, target_col
    
    def run_pipeline(self, data_path: str, n_iter: int = 100) -> ScientificLightGBM:
        """
        Run the complete training pipeline with Random Search.
        
        Parameters:
        -----------
        data_path : str
            Path to the data file
        n_iter : int
            Number of random search iterations (default: 100)
            - 100 iterations: ~1,500 fits (fast, good for initial exploration)
            - 200 iterations: ~3,000 fits (balanced)
            - 500 iterations: ~7,500 fits (thorough, recommended for publication)
        """
        logger.info("=" * 80)
        logger.info("SCIENTIFIC LIGHTGBM - TOTAL CHLORIDE PREDICTION")
        logger.info("VERSION 2.0 - REPEATED CV + RANDOMIZED SEARCH")
        logger.info("=" * 80)
        logger.info("\nRANDOM SEARCH DISTRIBUTIONS:")
        logger.info("  n_estimators    : [50, 100, 150, 200, 250, 300]")
        logger.info("  learning_rate   : [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.25, 0.3]")
        logger.info("  num_leaves      : [10, 15, 20, 25, 31, 40, 50, 63, 80, 100]")
        logger.info("  max_depth       : [3, 4, 5, 6, 7, 8, 9, 10, 12, 15]")
        logger.info("  min_child_samples: [5, 10, 15, 20, 25, 30, 40, 50]")
        logger.info("  subsample       : [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]")
        logger.info("  colsample_bytree: [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]")
        logger.info("  reg_alpha       : [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]")
        logger.info("  reg_lambda      : [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]")
        
        total_fits = n_iter * 5 * 3
        logger.info(f"\n  Random iterations: {n_iter}")
        logger.info(f"  Total CV fits: {n_iter} × 5 × 3 = {total_fits}")
        logger.info(f"  ⚡ This is ~{int(19440/total_fits)}x faster than full grid search")
        logger.info("\nFEATURE IMPORTANCE:")
        logger.info("  Method: Permutation Importance (preferred for publication)")
        logger.info("  Also: LightGBM built-in importance (Gain and Split)")
        logger.info("  Repeats: 30")
        logger.info("=" * 80 + "\n")
        
        X, y, target_col = self.load_data(data_path)
        
        # Model configuration
        model_config = {
            'test_size': 0.20,
            'random_state': 42,
            'cv_folds': 5,
            'cv_repeats': 3,
            'n_iter': n_iter,
            'n_jobs': -1,
            'cv_scoring': 'neg_root_mean_squared_error',
            'n_bootstrap': 1000,
            'n_permutation_repeats': 30
        }
        
        self.model = ScientificLightGBM(
            model_config,
            output_dir=self.results_dir
        )
        
        # Train model (split is done inside and stored)
        self.model.train(X, y, n_iter=n_iter)
        
        # Generate plots (using stored splits)
        self.visualizer.generate_all_plots(self.model)
        
        # Save model
        self.model.save_model()
        
        # Save results
        self._save_results(self.model)
        
        return self.model
    
    def _save_results(self, model: ScientificLightGBM):
        """Save all results as CSV and JSON."""
        
        # Training metrics
        train_metrics_df = pd.DataFrame({
            'Metric': list(model.metrics_train.keys()),
            'Value': [model.metrics_train[m] for m in model.metrics_train.keys()]
        })
        train_metrics_df.to_csv(os.path.join(self.results_dir, 'Tables', 'training_metrics.csv'), index=False)
        
        # Test metrics
        metrics_df = pd.DataFrame({
            'Metric': list(model.metrics_test.keys()),
            'Value': [model.metrics_test[m] for m in model.metrics_test.keys()]
        })
        metrics_df.to_csv(os.path.join(self.results_dir, 'Tables', 'test_metrics.csv'), index=False)
        
        # CV metrics
        cv_metrics = {
            'R2_mean': model.metrics_cv['R2_stats']['mean'],
            'R2_std': model.metrics_cv['R2_stats']['std'],
            'R2_ci_lower': model.metrics_cv['R2_stats']['ci_lower'],
            'R2_ci_upper': model.metrics_cv['R2_stats']['ci_upper'],
            'RMSE_mean': model.metrics_cv['RMSE_stats']['mean'],
            'RMSE_std': model.metrics_cv['RMSE_stats']['std'],
            'RMSE_ci_lower': model.metrics_cv['RMSE_stats']['ci_lower'],
            'RMSE_ci_upper': model.metrics_cv['RMSE_stats']['ci_upper'],
            'n_folds': len(model.metrics_cv['fold_metrics'])
        }
        cv_df = pd.DataFrame([cv_metrics])
        cv_df.to_csv(os.path.join(self.results_dir, 'Tables', 'cv_metrics.csv'), index=False)
        
        # CV fold metrics
        if 'fold_metrics' in model.metrics_cv:
            fold_df = pd.DataFrame(model.metrics_cv['fold_metrics'])
            fold_df.to_csv(os.path.join(self.results_dir, 'Tables', 'cv_fold_metrics.csv'), index=False)
        
        # Test bootstrap CIs
        if model.metrics_test_bootstrap:
            bootstrap_rows = []
            for metric, values in model.metrics_test_bootstrap.items():
                bootstrap_rows.append({
                    'Metric': metric,
                    'Mean': values['mean'],
                    'Std': values['std'],
                    '2.5% CI': values['lower_95'],
                    '97.5% CI': values['upper_95']
                })
            bootstrap_df = pd.DataFrame(bootstrap_rows)
            bootstrap_df.to_csv(os.path.join(self.results_dir, 'Tables', 'bootstrap_ci_test.csv'), index=False)
        
        # CV bootstrap CIs
        if hasattr(model, 'metrics_cv_bootstrap'):
            bootstrap_rows = []
            for metric, values in model.metrics_cv_bootstrap.items():
                bootstrap_rows.append({
                    'Metric': metric,
                    'Mean': values['mean'],
                    'Std': values['std'],
                    '2.5% CI': values['lower_95'],
                    '97.5% CI': values['upper_95']
                })
            bootstrap_df = pd.DataFrame(bootstrap_rows)
            bootstrap_df.to_csv(os.path.join(self.results_dir, 'Tables', 'bootstrap_ci_cv.csv'), index=False)
        
        # Random search results
        if model.random_search_results is not None:
            cv_results_df = pd.DataFrame(model.random_search_results.cv_results_)
            cv_results_df.to_csv(os.path.join(self.results_dir, 'Tables', 'random_search_results.csv'), index=False)
        
        # Best parameters
        with open(os.path.join(self.results_dir, 'Reports', 'best_params.json'), 'w') as f:
            json.dump(model.best_params, f, indent=2)
        
        # Permutation importance
        importance_df = model.get_feature_importance(method='permutation')
        if importance_df is not None and not importance_df.empty:
            importance_df.to_csv(os.path.join(self.results_dir, 'Tables', 'permutation_importance.csv'), index=False)
        
        # LightGBM Gain importance
        gain_importance = model.get_feature_importance(method='gain')
        if not gain_importance.empty:
            gain_importance.to_csv(os.path.join(self.results_dir, 'Tables', 'lightgbm_gain_importance.csv'), index=False)
        
        # LightGBM Split importance
        split_importance = model.get_feature_importance(method='split')
        if not split_importance.empty:
            split_importance.to_csv(os.path.join(self.results_dir, 'Tables', 'lightgbm_split_importance.csv'), index=False)
        
        # Reproducibility
        if model.reproducibility_info:
            with open(os.path.join(self.results_dir, 'Reports', 'reproducibility.json'), 'w') as f:
                json.dump(model.reproducibility_info, f, indent=2)
        
        # Publication summary
        report_path = os.path.join(self.results_dir, 'Reports', 'paper_summary.txt')
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("=" * 80 + "\n")
            f.write("LIGHTGBM MODEL - PUBLICATION READY RESULTS\n")
            f.write("CHLORIDE INGRESS - TOTAL CHLORIDE PREDICTION\n")
            f.write("VERSION 2.0 - REPEATED CV + RANDOMIZED SEARCH\n")
            f.write("=" * 80 + "\n\n")
            
            f.write("DATA SPLIT:\n")
            f.write("-" * 40 + "\n")
            f.write(f"  Training: {model.n_train_samples} samples (80%)\n")
            f.write(f"  Testing : {model.n_test_samples} samples (20%)\n")
            f.write(f"  Features: {model.n_features}\n")
            
            f.write("\nHYPERPARAMETER OPTIMIZATION:\n")
            f.write("-" * 40 + "\n")
            f.write(f"  Method: RandomizedSearchCV\n")
            f.write(f"  Iterations: {model.config['n_iter']}\n")
            f.write(f"  CV: RepeatedKFold (5×3 = 15 folds)\n")
            f.write(f"  Total fits: {model.config['n_iter'] * 5 * 3}\n")
            
            f.write("\nTRAINING SET METRICS:\n")
            f.write("-" * 40 + "\n")
            for metric, value in model.metrics_train.items():
                f.write(f"  {metric:15s}: {value:.4f}\n")
            
            f.write("\nCROSS-VALIDATION (RepeatedKFold 5×3):\n")
            f.write("-" * 40 + "\n")
            f.write(f"  R²  : {model.metrics_cv['R2_stats']['mean']:.4f} ± {model.metrics_cv['R2_stats']['std']:.4f}\n")
            f.write(f"  R² 95% CI: [{model.metrics_cv['R2_stats']['ci_lower']:.4f}, {model.metrics_cv['R2_stats']['ci_upper']:.4f}]\n")
            f.write(f"  RMSE: {model.metrics_cv['RMSE_stats']['mean']:.4f} ± {model.metrics_cv['RMSE_stats']['std']:.4f}\n")
            f.write(f"  RMSE 95% CI: [{model.metrics_cv['RMSE_stats']['ci_lower']:.4f}, {model.metrics_cv['RMSE_stats']['ci_upper']:.4f}]\n")
            f.write(f"  Number of folds: {len(model.metrics_cv['fold_metrics'])}\n")
            
            f.write("\nTEST SET METRICS:\n")
            f.write("-" * 40 + "\n")
            for metric, value in model.metrics_test.items():
                f.write(f"  {metric:15s}: {value:.4f}\n")
            
            f.write("\nBOOTSTRAP 95% CI (Test Set):\n")
            f.write("-" * 40 + "\n")
            for metric, values in model.metrics_test_bootstrap.items():
                f.write(f"  {metric:15s}: {values['mean']:.4f} [{values['lower_95']:.4f}, {values['upper_95']:.4f}]\n")
            
            f.write("\nBOOTSTRAP 95% CI (Cross-Validation):\n")
            f.write("-" * 40 + "\n")
            for metric, values in model.metrics_cv_bootstrap.items():
                f.write(f"  {metric:15s}: {values['mean']:.4f} [{values['lower_95']:.4f}, {values['upper_95']:.4f}]\n")
            
            f.write("\nBEST HYPERPARAMETERS:\n")
            f.write("-" * 40 + "\n")
            for param, value in model.best_params.items():
                clean_param = param.replace('model__', '')
                f.write(f"  {clean_param:25s}: {value}\n")
            
            f.write("\nPERMUTATION IMPORTANCE (Top 10):\n")
            f.write("-" * 40 + "\n")
            importance_df = model.get_feature_importance(method='permutation')
            if importance_df is not None and not importance_df.empty:
                for i, row in importance_df.head(10).iterrows():
                    f.write(f"  {row['Feature']:30s}: {row['Importance_mean']:.4f} ± {row['Importance_std']:.4f}\n")
            
            f.write("\nLIGHTGBM GAIN IMPORTANCE (Top 10):\n")
            f.write("-" * 40 + "\n")
            gain_importance = model.get_feature_importance(method='gain')
            if not gain_importance.empty:
                for i, row in gain_importance.head(10).iterrows():
                    f.write(f"  {row['Feature']:30s}: {row['Importance']:.4f}\n")
            
            f.write("\nRESIDUAL ANALYSIS (Test Set):\n")
            f.write("-" * 40 + "\n")
            for metric, value in model.residual_analysis['test'].items():
                if metric != 'shapiro_wilk_p' or not np.isnan(value):
                    f.write(f"  {metric:20s}: {value:.4f}\n")
            
            f.write("\nPREDICTIONS SAVED TO:\n")
            f.write("-" * 40 + "\n")
            f.write(f"  Predictions: {os.path.join(self.results_dir, 'Predictions')}\n")
            
            f.write("\n" + "=" * 80 + "\n")
        
        logger.info(f"All results saved to: {self.results_dir}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    print("\n" + "=" * 80)
    print("SCIENTIFIC LIGHTGBM - TOTAL CHLORIDE PREDICTION")
    print("VERSION 2.0 - REPEATED CV + RANDOMIZED SEARCH")
    print("=" * 80)
    print("\nRANDOM SEARCH DISTRIBUTIONS:")
    print("  n_estimators    : [50, 100, 150, 200, 250, 300]")
    print("  learning_rate   : [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.25, 0.3]")
    print("  num_leaves      : [10, 15, 20, 25, 31, 40, 50, 63, 80, 100]")
    print("  max_depth       : [3, 4, 5, 6, 7, 8, 9, 10, 12, 15]")
    print("  min_child_samples: [5, 10, 15, 20, 25, 30, 40, 50]")
    print("  subsample       : [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]")
    print("  colsample_bytree: [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]")
    print("  reg_alpha       : [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]")
    print("  reg_lambda      : [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]")
    
    print("\nRANDOM SEARCH CONFIGURATION:")
    print("  Iterations: 100 (recommended for publication)")
    print("  CV: RepeatedKFold (5×3 = 15 folds)")
    print("  Total fits: 100 × 15 = 1,500")
    print("  ⚡ This is 13× faster than full grid search (19,440 fits)")
    print("\nFEATURE IMPORTANCE:")
    print("  Method 1: Permutation Importance (primary)")
    print("  Method 2: LightGBM Gain Importance")
    print("  Method 3: LightGBM Split Importance")
    print("  Repeats: 30")
    print("\nOUTPUT FILES:")
    print("  📊 17 Publication-quality plots")
    print("  📊 Training and test metrics (CSV)")
    print("  📊 Predictions with actual values (CSV)")
    print("  📊 Feature importance rankings (CSV)")
    print("  📊 Publication summary (TXT)")
    print("\n" + "=" * 80 + "\n")
    
    try:
        data_path = r"D:\2026 Work\My Papers\Chloride Ingress\Data\Data.csv"
        
        trainer = ScientificModelTrainer()
        
        # Choose number of random iterations:
        # - 100: Fast (~1,500 fits) - good for initial exploration
        # - 200: Balanced (~3,000 fits)
        # - 500: Thorough (~7,500 fits) - recommended for publication
        model = trainer.run_pipeline(data_path, n_iter=100)
        
        print("\n" + "=" * 80)
        print("FINAL RESULTS SUMMARY")
        print("=" * 80)
        
        print(f"\nData Split:")
        print(f"  Training: {model.n_train_samples} samples (80%)")
        print(f"  Testing : {model.n_test_samples} samples (20%)")
        print(f"  Features: {model.n_features}")
        
        print(f"\nTraining Set Performance:")
        for metric, value in model.metrics_train.items():
            print(f"  {metric:15s}: {value:.4f}")
        
        print(f"\nCross-Validation (RepeatedKFold 5×3 = 15 folds):")
        print(f"  R²  : {model.metrics_cv['R2_stats']['mean']:.4f} ± {model.metrics_cv['R2_stats']['std']:.4f}")
        print(f"  R² 95% CI: [{model.metrics_cv['R2_stats']['ci_lower']:.4f}, {model.metrics_cv['R2_stats']['ci_upper']:.4f}]")
        print(f"  RMSE: {model.metrics_cv['RMSE_stats']['mean']:.4f} ± {model.metrics_cv['RMSE_stats']['std']:.4f}")
        print(f"  RMSE 95% CI: [{model.metrics_cv['RMSE_stats']['ci_lower']:.4f}, {model.metrics_cv['RMSE_stats']['ci_upper']:.4f}]")
        
        print(f"\nTest Set Performance:")
        for metric, value in model.metrics_test.items():
            print(f"  {metric:15s}: {value:.4f}")
        
        print(f"\nBootstrap 95% CI (Test Set - R²):")
        print(f"  {model.metrics_test_bootstrap['R2']['mean']:.4f} "
              f"[{model.metrics_test_bootstrap['R2']['lower_95']:.4f}, "
              f"{model.metrics_test_bootstrap['R2']['upper_95']:.4f}]")
        
        print(f"\nBootstrap 95% CI (CV - R²):")
        print(f"  {model.metrics_cv_bootstrap['R2']['mean']:.4f} "
              f"[{model.metrics_cv_bootstrap['R2']['lower_95']:.4f}, "
              f"{model.metrics_cv_bootstrap['R2']['upper_95']:.4f}]")
        
        print(f"\nBest Parameters:")
        for param, value in model.best_params.items():
            clean_param = param.replace('model__', '')
            print(f"  {clean_param:25s}: {value}")
        
        print(f"\nPermutation Importance (Top 5):")
        importance_df = model.get_feature_importance(method='permutation')
        if importance_df is not None and not importance_df.empty:
            for i, row in importance_df.head(5).iterrows():
                print(f"  {row['Feature']:30s}: {row['Importance_mean']:.4f} ± {row['Importance_std']:.4f}")
        
        print(f"\nLightGBM Gain Importance (Top 5):")
        gain_importance = model.get_feature_importance(method='gain')
        if not gain_importance.empty:
            for i, row in gain_importance.head(5).iterrows():
                print(f"  {row['Feature']:30s}: {row['Importance']:.4f}")
        
        print(f"\nPredictions saved to: {os.path.join(trainer.results_dir, 'Predictions')}")
        print(f"Results saved to: {trainer.results_dir}")
        print("=" * 80)
        
    except Exception as e:
        logger.error(f"Training failed: {e}")
        import traceback
        traceback.print_exc()
        raise

if __name__ == "__main__":
    main()

2026-08-07 15:19:13,499 - INFO - ================================================================================
2026-08-07 15:19:13,500 - INFO - SCIENTIFIC LIGHTGBM - TOTAL CHLORIDE PREDICTION
2026-08-07 15:19:13,501 - INFO - VERSION 2.0 - REPEATED CV + RANDOMIZED SEARCH
2026-08-07 15:19:13,501 - INFO - ================================================================================
2026-08-07 15:19:13,502 - INFO - 
RANDOM SEARCH DISTRIBUTIONS:
2026-08-07 15:19:13,503 - INFO -   n_estimators    : [50, 100, 150, 200, 250, 300]
2026-08-07 15:19:13,504 - INFO -   learning_rate   : [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.25, 0.3]
2026-08-07 15:19:13,504 - INFO -   num_leaves      : [10, 15, 20, 25, 31, 40, 50, 63, 80, 100]
2026-08-07 15:19:13,505 - INFO -   max_depth       : [3, 4, 5, 6, 7, 8, 9, 10, 12, 15]
2026-08-07 15:19:13,506 - INFO -   min_child_samples: [5, 10, 15, 20, 25, 30, 40, 50]
2026-08-07 15:19:13,506 - INFO -   subsample       : [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0


SCIENTIFIC LIGHTGBM - TOTAL CHLORIDE PREDICTION
VERSION 2.0 - REPEATED CV + RANDOMIZED SEARCH

RANDOM SEARCH DISTRIBUTIONS:
  n_estimators    : [50, 100, 150, 200, 250, 300]
  learning_rate   : [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.25, 0.3]
  num_leaves      : [10, 15, 20, 25, 31, 40, 50, 63, 80, 100]
  max_depth       : [3, 4, 5, 6, 7, 8, 9, 10, 12, 15]
  min_child_samples: [5, 10, 15, 20, 25, 30, 40, 50]
  subsample       : [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
  colsample_bytree: [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
  reg_alpha       : [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
  reg_lambda      : [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]

RANDOM SEARCH CONFIGURATION:
  Iterations: 100 (recommended for publication)
  CV: RepeatedKFold (5×3 = 15 folds)
  Total fits: 100 × 15 = 1,500
  ⚡ This is 13× faster than full grid search (19,440 fits)

FEATURE IMPORTANCE:
  Method 1: Permutation Importance (primary)
  Method 2: LightGBM Gain Impor

2026-08-07 15:22:42,113 - INFO - 
RandomizedSearchCV completed in 208.5 seconds
2026-08-07 15:22:42,115 - INFO - Best parameters: {'model__subsample': 0.8, 'model__reg_lambda': 5.0, 'model__reg_alpha': 0.1, 'model__num_leaves': 15, 'model__n_estimators': 250, 'model__min_child_samples': 5, 'model__max_depth': 7, 'model__learning_rate': 0.15, 'model__colsample_bytree': 0.9}
2026-08-07 15:22:42,116 - INFO - Best CV RMSE: 0.4249
2026-08-07 15:22:42,118 - INFO - 
Calculating cross-validation metrics...
2026-08-07 15:22:45,134 - INFO - 
Cross-Validation Metrics (5×3 = 15 folds):
2026-08-07 15:22:45,136 - INFO -   R²  : 0.8563 ± 0.0352
2026-08-07 15:22:45,137 - INFO -   R² 95% CI: [0.8125, 0.9000]
2026-08-07 15:22:45,139 - INFO -   RMSE: 0.5072 ± 0.0858
2026-08-07 15:22:45,140 - INFO -   RMSE 95% CI: [0.4006, 0.6138]
2026-08-07 15:22:45,141 - INFO - 
Retraining best model on full training set...
2026-08-07 15:22:45,178 - INFO - 
Training Set Metrics:
2026-08-07 15:22:45,179 - INFO -   R2    


FINAL RESULTS SUMMARY

Data Split:
  Training: 734 samples (80%)
  Testing : 184 samples (20%)
  Features: 15

Training Set Performance:
  R2             : 0.9887
  RMSE           : 0.1441
  MAE            : 0.1016
  MAPE           : 1828038748864.1741
  NRMSE_mean     : 0.0857
  Pearson        : 0.9944
  Spearman       : 0.9930

Cross-Validation (RepeatedKFold 5×3 = 15 folds):
  R²  : 0.8563 ± 0.0352
  R² 95% CI: [0.8125, 0.9000]
  RMSE: 0.5072 ± 0.0858
  RMSE 95% CI: [0.4006, 0.6138]

Test Set Performance:
  R2             : 0.9274
  RMSE           : 0.3533
  MAE            : 0.2379
  MAPE           : 1.3995
  NRMSE_mean     : 0.2122
  NRMSE_range    : 0.0576
  Pearson        : 0.9633
  Spearman       : 0.9680

Bootstrap 95% CI (Test Set - R²):
  0.9265 [0.9007, 0.9480]

Bootstrap 95% CI (CV - R²):
  0.8562 [0.8274, 0.8802]

Best Parameters:
  subsample                : 0.8
  reg_lambda               : 5.0
  reg_alpha                : 0.1
  num_leaves               : 15
  n_estimato